In [0]:
# MAGIC ## 1. Setup
from pyspark.sql import functions as F

SOURCE_PATH = "/databricks-datasets/retail-org/products/"
TARGET_TABLE = "retail_dev.bronze.bronze_products"


In [0]:
# MAGIC ## 2. Schema e Volume
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")
    .csv(SOURCE_PATH)
)

print("=== SCHEMA ===")
df.printSchema()

print(f"\n=== VOLUME ===")
print(f"Total de linhas:   {df.count()}")
print(f"Total de colunas:  {len(df.columns)}")
print(f"Colunas:           {df.columns}")


In [0]:
# MAGIC ## 3. Amostra
display(df.limit(10))


In [0]:
# MAGIC ## 4. Qualidade — Nulos por Coluna
display(
    df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])
)
# MAGIC ### Produtos duplicados?
display(
    df.groupBy("product_id")
    .agg(F.count("*").alias("ocorrencias"))
    .filter(F.col("ocorrencias") > 1)
    .orderBy(F.col("ocorrencias").desc())
)

In [0]:
# MAGIC ## 5. Análise
# MAGIC ### Distribuição por categoria
display(
    df.groupBy("product_category")
    .agg(F.count("*").alias("total_produtos"))
    .orderBy(F.col("total_produtos").desc())
)
# MAGIC ### Estatísticas de preço (`sales_price`)
display(
    df.select(
        F.count("sales_price").alias("total_com_preco"),
        F.sum(F.when(F.col("sales_price").isNull(), 1).otherwise(0)).alias("sem_preco"),
        F.min("sales_price").alias("preco_minimo"),
        F.max("sales_price").alias("preco_maximo"),
        F.avg("sales_price").alias("preco_medio"),
    )
)

# MAGIC ### Distribuição por unidade de medida (`product_unit`)

display(
    df.groupBy("product_unit")
    .agg(F.count("*").alias("total"))
    .orderBy(F.col("total").desc())
)

In [0]:
# MAGIC ## 6. Ingestão → Delta
# MAGIC Escrita na camada bronze — estrutura raw preservada, zero transformações.
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Tabela escrita: {TARGET_TABLE}")

In [0]:

# MAGIC ## 7. Verificação
df_check = spark.table(TARGET_TABLE)

print(f"Linhas na tabela: {df_check.count()}")
print(f"Colunas:          {df_check.columns}")

display(df_check.limit(5))
